In [ ]:
# !pip install accelerate bitsandbytes psutil
# !pip install timm

In [ ]:
from transformers import AutoProcessor, Gemma3nForConditionalGeneration, BitsAndBytesConfig, AutoModelForCausalLM
import torch
import librosa
import numpy as np

In [ ]:
# ##############################
# #Memory cleaning

# import torch
# import gc

# torch.cuda.empty_cache()
# gc.collect() # python garbage collector
# ##############################

In [ ]:
model_id = "google/gemma-3n-e2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # nf4 instead of bf4, cpu error
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Gemma3nForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float32, # bfloat16 does not work, float16 does not work either, only float32
    device_map="auto",
    trust_remote_code=True
).eval()

In [ ]:
model.gradient_checkpointing_enable()
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
"""
RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.cuda.HalfTensor) should be the same
"""

# Fix:
# float32 instead of float16

"""
RuntimeError: Expected all tensors to be on the same device,
but found at least two devices, cuda:0 and cpu! (when checking argument for argument min in method wrapper_CUDA_clamp_Tensor)
"""

# Fix:
# llm_int8_enable_fp32_cpu_offload=True -> does not work, either more quantization, or more gpu memory


In [ ]:
def process_audio_file(audio_path,
                       prompt= """Ты - эксперт по греческому языку. Послушай аудиозапись и создай точную транскрипцию.
Сохрани:
- Оригинальные греческие слова
- Естественную пунктуацию
- Структуру предложений

Транскрипция:"""
                       ):
    audio, sr = librosa.load(audio_path, sr=16000)

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "Ты полезный ассистент, который может анализировать аудио."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    generation = generation[0][input_len:]
    decoded = processor.decode(generation, skip_special_tokens=True)

    return decoded

In [ ]:
process_audio_file(audio_path="./chunk_2.wav")